SHAP = Açıklanabilir AI. Gürültüyü azalmak için anomalileri tespit etmemiz lazım. Tespit etmek de yeterli değildir. Direkt modelin anomali belirlediği pencereye bakarak hangi sensörden dolayı anomali belirlediği, bunların birbiri ile ilişkileri ve modelin karar verirken ki ağırlıklarının incelenmesi gerekecek.

Örneğin:
Kayıt #45231 → Anomali skoru yüksek
 -- SHAP değerleri:
 -- Oil_temperature_diff_1h  : +2.3  → Anomaliye en çok katkı yapan
 -- Motor_current_rmean_6h   : +1.8  → İkinci en önemli
 -- TP2_lag_1h               : +0.9  → Üçüncü
 -- TP3_rmean_24h            : -0.3  → Anomali olmadığına işaret ediyor

 Model neden anomali dedi? Hangi sensör kararı tetikledi? Bunu görsel olarak analiz edeceğiz. İş birim metrikleri sensörleri inceleyeceğiz. Anomali skorlarını zaman serisinde görselleştireceğiz.

In [ ]:
""" DOSYA YOLLARI """
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

kaggle da upload edilen dosyaların yol kontrolleri

In [ ]:
""" MODEL VE VERİLERİ YÜKLEME """
import polars as pl
import numpy as np
import pickle
import tensorflow as tf

# Yollar
FEATURES_PATH = "/kaggle/input/datasets/denizbyat/fatures-set/metropt3_features.parquet"        # özelliklerin bulunduğu Parquet dosyası
MODEL_PATH = "/kaggle/input/datasets/denizbyat/metropt3-features-data/lstm_autoencoder.keras"   # eğitilmiş LSTM model dosyası
SCALER_PATH = "/kaggle/input/datasets/denizbyat/metropt3-features-data/scaler.pkl"              # veri normalize edecek ölçekleyici dosyası
THRESHOLD_PATH = "/kaggle/input/datasets/denizbyat/metropt3-features-data/lstm_threshold.npy"   # eşik değeri dosyası

# Özellik dosyasını Yükle
df = pl.read_parquet(FEATURES_PATH)

# eğitimde kullanılan ölçekleyiciyi geri yüklüyoruz ki veriyi aynı şekilde ölçekleyebilelim yoksa model beklediği formatta veri alamaz ve sonuçlar anlamsız olur
with open(SCALER_PATH, "rb") as f:      # "rb" modunda açıyoruz çünkü pickle ikili formatta çalışır (bilgisayarın anlayacağı dilden 0-1)
    scaler = pickle.load(f)     # pickle.load() fonksiyonu ile dosyadan ölçekleyiciyi geri yüklüyoruz

autoencoder = tf.keras.models.load_model(MODEL_PATH)
threshold = np.load(THRESHOLD_PATH)

print(f"Veri: {df.shape[0]:,} satır, {df.shape[1]} sütun")
print(f"Eşik değeri: {threshold:.4f}")
print("Model yüklendi ✓")

Model, Veriler ve anomali tespiti için gerekli dosyalar yüklendi

In [ ]:
""" VERİ HAZIRLIĞI """
# Feature kolonları
feature_cols = [col for col in df.columns 
                if col not in ['timestamp', 'is_suspect']]  # timestamp ve is_suspect hariç tüm kolonlar özellik olarak kullanılacak

X = df.select(feature_cols).to_numpy()  # model girdileri
y = df['is_suspect'].to_numpy()     # hedef etiket (0: normal, 1: şüpheli)

# daha önce normalize ettiğimiz için sadece ölçekleyiciyi kullanarak bunu uyguluyoruz (bi yandan da veri sızıntıısı olmaması için iyi uygulamadır önceden eğitmiş olduğumuz scaler'ı kullanarak aynı şekilde ölçekleme yapıyoruz)
X_scaled = scaler.transform(X)

print(f"Feature matrix : {X_scaled.shape}")
print(f"Normal kayıt   : {(y == 0).sum():,}")
print(f"Şüpheli kayıt  : {(y == 1).sum():,}")

Sonuçlar böyledi şimdi bunların SHAP yani karar alımındaki ağırlık analizini yapıcaz Feature matrix : (1508308, 131) -- Normal kayıt   : 1,406,713 -- Şüpheli kayıt  : 101,595